<a href="https://colab.research.google.com/github/suchirsalhan/bilingual-l1-effects/blob/main/Prompting_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "lgb35/B-GPT-es-fineweb-beginner-alpacaEnglish"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

# Create generator
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/961 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/118 [00:00<?, ?B/s]

In [2]:
prompts = [
    {
        "prompt_id": "A1",
        "prompt_style": "basic",
        "prompt_topic": "hobbies",
        "instruction": "Your town wants people to take part in different activities and hobbies. Write an article for your local newspaper describing the activities you can do in your town at the weekend."
    },
    {
        "prompt_id": "B1",
        "prompt_style": "basic",
        "prompt_topic": "holiday",
        "instruction": "Your friend, Sophia, has emailed you about her recent holiday. Write an email to her, describing your favourite holiday you have been on."
    },
    {
        "prompt_id": "C1",
        "prompt_style": "basic",
        "prompt_topic": "family",
        "instruction": "Your local magazine is holding a competition for articles about the question ‘Big family or small family’. Write an article about your family. You should include whether you have a big or small family, and which you prefer."
    },
    { ###UPDATE FOR L1###
        "prompt_id": "A2",
        "prompt_style": "mixed",
        "prompt_topic": "hobbies",
        "instruction": "Your town wants people to take part en diversas actividades y aficiones. Write an article para tu periódico local describing the activities you can do in your town durante el fin de semana."
    },
    {
        "prompt_id": "B2",
        "prompt_style": "mixed",
        "prompt_topic": "holiday",
        "instruction": "Tu amiga, Sophia, has emailed you about her recent holiday. Write an email to her, describiéndole tus vacaciones favoritas."
    },
    {
        "prompt_id": "C2",
        "prompt_style": "mixed",
        "prompt_topic": "family",
        "instruction": "Tu revista local is holding a competition for articles about the question ‘Big family or small family’. Escribe un artículo sobre tu familia. You should include whether you have a big or small family, y cuál prefieres."
    },
    { ###UPDATE FOR L1###
        "prompt_id": "A3",
        "prompt_style": "l2t",
        "prompt_topic": "hobbies",
        "instruction": "Your town wants people to take part in different activities and hobbies. Write an article for your local newspaper describing the activities you can do in your town at the weekend. Think in Spanish."
    },
    {
        "prompt_id": "B3",
        "prompt_style": "l2t",
        "prompt_topic": "holiday",
        "instruction": "Your friend, Sophia, has emailed you about her recent holiday. Write an email to her, describing your favourite holiday you have been on. Think in Spanish."
    },
    {
        "prompt_id": "C3",
        "prompt_style": "l2t",
        "prompt_topic": "family",
        "instruction": "Your local magazine is holding a competition for articles about the question ‘Big family or small family’. Write an article about your family. You should include whether you have a big or small family, and which you prefer. Think in Spanish."
    }
]

In [3]:
results = []

for p in prompts:

    full_prompt = (
        "### Instruction:\n"
        f"{p['instruction']}\n\n"
        "### Response:\n"
    )

    output = generator(
        full_prompt,
        max_new_tokens=300,
        min_new_tokens=250,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = output[0]["generated_text"].replace(full_prompt, "").strip()

    results.append({
        "model_name": model_name,
        "prompt_id": p["prompt_id"],
        "prompt_style": p["prompt_style"],
        "prompt_topic": p["prompt_topic"],
        "instruction": p["instruction"],
        "generated_text": generated_text,
        "temperature": 0.7,
        "max_new_tokens": 300
    })

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'do_sample', 'temperature', 'min_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=250) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co

In [5]:
import json

filename = "B-GPT-es-fineweb-beginner-alpacaEnglish"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Saved to {filename}")

Saved to B-GPT-es-fineweb-beginner-alpacaEnglish
